### Sistema Recomendación Híbrido

El objetivo de este proyecto es construir un sistema de recomendación basado en las compras realizadas por los clientes de una superficie comercial. El enfoque parte de una restricción clave: al tratarse de compra física, no es posible recomendar en tiempo real durante la compra, ya que no conocemos la cesta del cliente hasta que pasa por caja. Esto diferencia el problema del de una tienda online, donde sí se puede recomendar según la cesta se va componiendo.
Por ello, el sistema se orienta a la recomendación entre visitas: a partir del historial de compra de cada cliente, generar recomendaciones personalizadas que incentiven su siguiente visita (mediante email, app o cupones en el ticket), en línea con los programas de fidelización habituales en el sector.
Para ello se construyen tres modelos:

- ALS, que aprende los hábitos de cada cliente a partir de su historial para predecir y recomendar su próxima compra. Es el núcleo de la personalización.
- Apriori, que analiza la composición de las cestas para extraer patrones de co-compra. Más que recomendar al cliente directamente, alimenta decisiones de negocio: colocación de productos en tienda, promociones cruzadas y cupones post-compra.
- Popularidad, que recomienda los productos más vendidos como red de seguridad para clientes nuevos o sin historial suficiente.

### Librerías

In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"

In [2]:
import pandas as pd
import numpy as np
from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix

In [3]:
df = pd.read_pickle('data/transacciones_limpio.pkl')

df.head(5)

,line_id,ticket_number,user_card_id,payment_method,shop_name,shop_address,product_code,product_name,seccion,units,price_per_unit,price_total,created_at
0,9550698,6569586,100552,EFECTIVO,Carrefour A Coruña Marineda,"C/ Carretera Baños de Arteixo 43, A Coruña",1041,HELADO VAINILLA 1L,Congelados,1,3.41,3.41,2025-01-01 08:20:00
1,9550697,6569586,100552,EFECTIVO,Carrefour A Coruña Marineda,"C/ Carretera Baños de Arteixo 43, A Coruña",1008,AZUCAR BLANCO 1KG,Alimentacion seca,1,1.06,1.06,2025-01-01 08:20:00
2,9550707,6569586,100552,EFECTIVO,Carrefour A Coruña Marineda,"C/ Carretera Baños de Arteixo 43, A Coruña",1055,BOLSAS BASURA 30L,Drogueria,1,1.98,1.98,2025-01-01 08:20:00
3,9550706,6569586,100552,EFECTIVO,Carrefour A Coruña Marineda,"C/ Carretera Baños de Arteixo 43, A Coruña",1022,CARNE PICADA MIXTA 500G,Frescos,1,3.95,3.95,2025-01-01 08:20:00
4,9550693,6569586,100552,EFECTIVO,Carrefour A Coruña Marineda,"C/ Carretera Baños de Arteixo 43, A Coruña",1068,ARENA GATO 5L,Mascotas,2,3.94,7.88,2025-01-01 08:20:00


Comprobamos longitud datos y fecha mínima y máxima para realizar división de datos

In [4]:
print(f'Filas: {len(df):,}')
print(df.created_at.min())
print(df.created_at.max())

Filas: 50,000
2025-01-01 08:20:00
2025-06-30 21:21:00


Tras comprobar fechas procedemos a separar los datos y crear un conjunto de entrenamiento y testeo. Esta separación se realiza de forma manual.

In [5]:
# Fecha corte. Últimos 30 días para test
fecha_corte = df.created_at.max() - pd.Timedelta(days=30)
print(f'Fecha corte: {fecha_corte}')

# creamos conjuntos
train = df[df.created_at <= fecha_corte].copy()
test = df[df.created_at > fecha_corte].copy()

print(f'Longitud Train: {len(train)}; ({len(train)/len(df)*100:.0f}%)') 
print(f'Longitud Test: {len(test)}; ({len(test)/len(df)*100:.0f}%)')
print('-'*50)
print(f'Tickets train: {train.ticket_number.nunique():,}') # comprobación de tickets por conjunto
print(f'Tickets test: {test.ticket_number.nunique():,}')


Fecha corte: 2025-05-31 21:21:00
Longitud Train: 41649; (83%)
Longitud Test: 8351; (17%)
--------------------------------------------------
Tickets train: 4,340
Tickets test: 856


Ahora vamos a comprobar el número de clientes evaluables. Este paso es importante porque determina la frontera a partir de la cual entra en juego cada modelo. Para ALS, un sistema personalizado, se necesita un registro histórico del cliente por tanto no funcionará con aquellos clientes nuevos, lo que se conoce como cold start. Estos clientes de cold start serán asignados a sistemas como Apriori o popularidad para realizar la recomendación.

In [6]:
# almacenamos los clientes en conjuntos y eliminamos duplicados
clientes_train = set(train.user_card_id.unique())
clientes_test = set(test.user_card_id.unique())

evaluables = clientes_test & clientes_train # intersección que guarda solo clientes que se encuentran en ambos conjuntos. Aparecen tanto en train como test
cold_start = clientes_test - clientes_train  # clientes que compraron por primera vez dentro del rango de test.

print(f'Clientes en test: {len(clientes_test)}')
print(f'Evaluables: {len(evaluables)}') # clientes sobre los que se puede probar ALS
print(f'Cold start puro (test): {len(cold_start)}')

Clientes en test: 583
Evaluables: 537
Cold start puro (test): 46


Con estos cálculos ya tenemos una idea sobre la cantidad de clientes que vamos a manejar en las evaluaciones de modelos.
Contamos con 583 clientes dentro del conjunto de testeo de los cuales, 46 de ellos han realizado su primera compra en el periodo que contempla el conjunto de test. Los restantes 537 clientes se encuentran en ambos conjuntos y nos ayudarán a medir si ALS acierta. 

### Modelo Basado en Popularidad

El primer modelo que se construye es un sistema NO personalizado. Este modelo solo se va a fijar en cuantas veces aparece cada producto en cada ticket. Se trata de un modelo totalmente sencillo que no necesita conocer ningún cliente, es ideal para cuando estamos empezando o el cliente es nuevo, se usa como referencia base.

In [7]:
# Popularidad de producto según tickets en conjunto train
popularidad = (
    train.groupby('product_name')['ticket_number']
    .nunique()
    .sort_values(ascending=False)
)

print('Top 10 productos más populares (por nº de tickets):')
print(popularidad.head(10))

Top 10 productos más populares (por nº de tickets):
product_name
CHAMPU ANTICASPA 400ML      3393
LECHUGA ICEBERG UD          2843
YOGUR NATURAL PACK 4        2231
HELADO VAINILLA 1L          1855
ARROZ REDONDO 1KG           1643
TOMATE RAMA KG              1610
ENSALADA CESAR PREPARADA    1336
CERVEZA PACK 6              1264
SAL FINA 1KG                1117
ARENA GATO 5L               1073
Name: ticket_number, dtype: int64


In [8]:
def recomendar_popularidad(n=10):
    """
    Función para recomendar productos en base a la popularidad. Veces que aparece en tickets
    Parámetros:
    -n: número de productos seleccionados
    """
    return popularidad.head(n).index.tolist()

print(recomendar_popularidad(10))

['CHAMPU ANTICASPA 400ML', 'LECHUGA ICEBERG UD', 'YOGUR NATURAL PACK 4', 'HELADO VAINILLA 1L', 'ARROZ REDONDO 1KG', 'TOMATE RAMA KG', 'ENSALADA CESAR PREPARADA', 'CERVEZA PACK 6', 'SAL FINA 1KG', 'ARENA GATO 5L']


Vamos a evaluar el comportamiento de este sistema base

In [9]:
recomendados = set(popularidad.head(10).index) # creamos un conjunto con los 10 productos más populares

# Comprobamos cliente de forma individual
cliente = list(evaluables)[0] # creamos una lista con los clientes y seleccionamos el primer cliente
comprados = set(test[test['user_card_id'] == cliente]['product_name']) # conjunto de productos comprados por el cliente dentro de test

print(f'Cliente: {cliente}')
print(f'Productos comprados en conjunto test: {len(comprados)}')
print(comprados)

Cliente: 100352
Productos comprados en conjunto test: 23
{'TOMATE RAMA KG', 'ENSALADA CESAR PREPARADA', 'REFRESCO COLA 2L', 'MANZANA GOLDEN KG', 'HUEVOS DOCENA M', 'PIZZA CUATRO QUESOS', 'ARROZ REDONDO 1KG', 'PECHUGA POLLO BANDEJA', 'TOMATE FRITO 400G', 'MERLUZA FILETE KG', 'LECHUGA ICEBERG UD', 'MACARRONES 500G', 'LECHE INFANTIL CONTINUACION', 'CHAMPU ANTICASPA 400ML', 'SAL FINA 1KG', 'YOGUR NATURAL PACK 4', 'LECHE ENTERA BRIK 1L', 'AZUCAR BLANCO 1KG', 'CAFE MOLIDO 250G', 'PATATA KG', 'HELADO VAINILLA 1L', 'ARENA GATO 5L', 'PAPILLA CEREALES 600G'}


Guardamos en una variable los productos comprados por un cliente concreto durante el periodo de testeo. Ahora vamos a comparar este cliente con el top de productos más populares y ver la tasa de acierto.

In [10]:
aciertos = recomendados & comprados # creamos una intersección entre productos recomendados y comprados del cliente seleccionado

print(f'Nº de aciertos: {len(aciertos)}')
print(f'Aciertos: {aciertos}')

Nº de aciertos: 9
Aciertos: {'TOMATE RAMA KG', 'ENSALADA CESAR PREPARADA', 'HELADO VAINILLA 1L', 'LECHUGA ICEBERG UD', 'CHAMPU ANTICASPA 400ML', 'SAL FINA 1KG', 'ARENA GATO 5L', 'YOGUR NATURAL PACK 4', 'ARROZ REDONDO 1KG'}


De los 23 productos comprados por el cliente en el conjunto de test, 9 de ellos se encuentran dentro de los 10 más populares

In [11]:
precision = len(aciertos) / 10 # cuantos valores acertamos
recall = len(aciertos) / len(comprados) # de los productos comprados, cuantos cubrimos

print(f'Precision@10 para el cliente: {precision:.2f}')
print(f'Recall@10 para el cliente: {recall:.2f}')

Precision@10 para el cliente: 0.90
Recall@10 para el cliente: 0.39


La precision alta es esperable en un baseline de popularidad cuando los clientes consumen mayoritariamente productos populares. Por el contrario, el recall queda limitado de forma estructural, al recomendar solo 10 productos de 23 que compra este cliente, el valor máximo que se puede alcanzar es 0.43, siendo este resultado una métrica comparativa entre modelos.

In [12]:
# inicializamos lista para almacenar resultados de todos los clientes
precisiones = []
recalls = []

# inicializamos bucle
for c in evaluables:
    productos_comprados = set(test[test['user_card_id'] == c]['product_name'])

    productos_acertados = len(recomendados & productos_comprados) # intersección entre productos comprados y top 10 más populares.

    precisiones.append(productos_acertados/10) # añadimos la precision de cada cliente al listado
    recalls.append(productos_acertados/ len(productos_comprados)) # añadimos el recall de cada cliente al listado.

print(f'Cliente evaluados: {len(precisiones)}')
print(f'Precision@10 media : {np.mean(precisiones):.4f}')
print(f'Recall@10 media : {np.mean(recalls):.4f}')


Cliente evaluados: 537
Precision@10 media : 0.5058
Recall@10 media : 0.4515


Tras realizar el mecanismo de forma individual para un solo cliente, replicamos lo mismo en bucle para todos los clientes que aparecen en train y test. Esto devuelve una precision media de 0.5058, es decir, 5 de cada 10 productos recomendados acaban siendo compras reales del cliente. Se puede decir que es un buen resultado para un modelo NO personalizado.

En cuanto al Recall, este indica que los productos acertados representan el 45% de lo que el cliente compró.

El rendimiento del modelo no personalizado obtiene unos resultados muy buenos que pueden ser difíciles de superar para un modelo basado en ALS. El principal motivo de estos resultados es el propio dataset, no existe una gran diferenciación de clientes y el número de productos que se maneja es pequeño en comparación a un entorno real. Aunque se trate de datos sintéticos, esto no implica que en un entorno real, siempre vaya a tener mejor resultado un modelo más complejo como ALS, para una superficie tipo supermercado, recomendar simplemente los productos más vendidos puede ser más eficaz que tratar de construir un modelo más complejo.

In [13]:
pop_train = set(popularidad.head(10).index)
pop_test = set(test.groupby('product_name')['ticket_number'].nunique()
               .sort_values(ascending=False).head(10).index)

print(f'Productos en común entre top-10 train y top-10 test: {len(pop_train & pop_test)}/10')

Productos en común entre top-10 train y top-10 test: 9/10


Por último, comprobamos como de estable es el top 10 de productos entre train y test. El resultado que se obtiene es que 9 de cada 10 productos coinciden entre ambos conjuntos. (SUBIR ESTO ARRIBA TIENE MÁS SENTIDO)

### Sistema ALS

Mientras que el modelo basado en popularidad es un sistema no personalizado, el sistema ALS es un sistema de recomendación personalizado, en donde cada cliente recibe una recomendación acorde a su perfil.

El método Alternating Least Squares (ALS) consiste en crear una matriz cliente-producto en donde se registran las veces que cada cliente ha comprado cada producto. Esa matriz será la capa de entrada al modelo y donde empieza la factorización, partiendo de esta matriz ALS la descompone en dos matrices más pequeñas, una de clientes y otra de productos, descritas por factores latentes. Los factores latentes son características ocultas que el modelo inventa para explicar los datos. Este factor latente es similar a un ajuste de parámetros en ML clásico. 

Con esto, ALS congela una matriz mientras resuelve la otra y una vez tiene una resuelta, repite el proceso para la que aún no está resuelta apoyandose en la otra matriz. Una vez se ha completado este proceso, se realizan las prediciones que consiste en multiplicar los vecoters de factores. Si estos vectores están alineados, la predicción es alta y se devuelve la predicción más alta como recomendación.

La idea de seleccionar este método frente a otros se encuentra en que en un entorno real, la matriz de cliente-producto es muy dispersa y por tanto un sistema item-item tendrá complicaciones para realizar las recomendaciones. En este ejercicio la matriz con la que se trabaja no es dispersa.

In [14]:
# Frecuencia, nº de veces que cada cliente compró cada producto

interacciones = (train.groupby(['user_card_id', 'product_name'])
                 .size()
                 .reset_index(name='frecuencia'))

print(f'Interaciones cliente-producto: {len(interacciones)}')
interacciones.head()

Interaciones cliente-producto: 26204


,user_card_id,product_name,frecuencia
0,100001,CARNE PICADA MIXTA 500G,1
1,100001,CEBOLLA KG,1
2,100001,CHAMPU ANTICASPA 400ML,2
3,100001,ENSALADA CESAR PREPARADA,1
4,100001,HELADO VAINILLA 1L,1


Como la librería **implicit** trabaja con matrices donde filas y columnas son índices enteros hay que aplicar un mapeo.

In [15]:
# lista única productos y clientes
clientes_unicos = interacciones['user_card_id'].unique()
productos_unicos = interacciones['product_name'].unique()

# Mapeo
cliente_idx = {c: i for i, c in enumerate(clientes_unicos)} # compresión diccionario
productos_idx = {p: i for i, p in enumerate(productos_unicos)} # compresión diccionario

# Mapeo inverso; interpretación de resultados posterior
idx_cliente = {i: c for c, i in cliente_idx.items()}
idx_producto = {i: p for p, i in productos_idx.items()}

print(f'Clientes: {len(clientes_unicos)}')
print(f'Productos: {len(productos_unicos)}')


Clientes: 1296
Productos: 68


Una vez mapeados clientes y productos es hora de construir la matriz que recibe ALS. La matriz dispersa solo almacena las celdas que contienen valores ignorando aquellas que son cero. Para estos datos, tal vez no se aprecia su efecto, pero para grandes superficies supone una diferencia grande en cuanto a tiempo.

In [16]:
# Asignamos valores mapeados a cada valor.
filas = interacciones['user_card_id'].map(cliente_idx)
columnas = interacciones['product_name'].map(productos_idx)
valores = interacciones['frecuencia'].astype(float)

# Matriz dispersa cliente x producto
matriz_cp = csr_matrix(
    (valores, (filas,columnas)),
    shape=(len(clientes_unicos), len(productos_unicos))
)

print(f'Forma matriz: {matriz_cp.shape}')
print(f'Valores NO nulos: {matriz_cp.nnz:}')
print(f'Densidad: {matriz_cp.nnz / (matriz_cp.shape[0]*matriz_cp.shape[1])*100:.2f}%')

Forma matriz: (1296, 68)
Valores NO nulos: 26204
Densidad: 29.73%


Es importante tener en cuenta que se construye la matriz sobre el conjunto TRAIN por tanto el número de clientes se reduce ya que no todos los clientes de TRAIN se encuentran en TEST. Este método solo puede aprender con aquellos clientes que tienen historial, por tanto los nuevos clientes, los que se clasifican como cold start quedan fuera de este modelo.

Con la matriz creada vamos a iterar con ALS por primera vez, los valores que van a recibir los parámetros son los propios valores predeterminados que vienen con el modelo, sin ningún tipo de ajuste. Posteriormente se ajustarán estos parámetros mediante Optuna.

In [17]:
# modelo base
modelo_base_als = AlternatingLeastSquares(random_state=42)

modelo_base_als.fit(matriz_cp)
print('Modelo base entrenado')

  0%|          | 0/15 [00:00<?, ?it/s]

Modelo base entrenado


In [20]:
cliente_real = clientes_unicos[0] # ID real del cliente
cliente_index = cliente_idx[cliente_real] # indice en matriz

# Generación de recomendaciones
ids, scores = modelo_base_als.recommend(
    cliente_index,
    matriz_cp[cliente_index], # fila del cliente en matriz
    N=10 # top 10
)

# Pasamos los índices de vuelta a nombres de producto
recomendaciones_als = [idx_producto[i] for i in ids]

print(f'Cliente: {cliente_real}')
print('Recomendaciones')
for prod, score in zip(recomendaciones_als, scores):
    print(f'{prod:35s} {score:.3f}')

Cliente: 100001
Recomendaciones
PIZZA JAMON Y QUESO                 0.012
CERVEZA PACK 6                      0.011
PIZZA CUATRO QUESOS                 0.010
REFRESCO NARANJA 2L                 0.009
REFRESCO COLA 2L                    0.009
LASAÑA BOLOÑESA 400G                0.008
ARENA GATO 5L                       0.008
VINO TINTO CRIANZA                  0.008
ARROZ REDONDO 1KG                   0.006
GUISANTES CONGELADOS 1KG            0.006


In [22]:
print("\nLo que compró en train:")
set(train[train["user_card_id"] == cliente_real]["product_name"])


Lo que compró en train:


{'CARNE PICADA MIXTA 500G',
 'CEBOLLA KG',
 'CHAMPU ANTICASPA 400ML',
 'ENSALADA CESAR PREPARADA',
 'HELADO VAINILLA 1L',
 'LECHUGA ICEBERG UD',
 'MANTEQUILLA 250G',
 'MERLUZA FILETE KG',
 'PAN BARRA RUSTICA',
 'PASTA ESPAGUETI 500G',
 'PATATA KG',
 'QUESO LONCHAS 200G',
 'SAL FINA 1KG',
 'SUAVIZANTE CONCENTRADO 1.5L',
 'TOMATE FRITO 400G',
 'TOMATE RAMA KG',
 'TORTILLA PATATA REFRIGERADA',
 'YOGUR NATURAL PACK 4'}